## **Tutorial 1. PLS Regression Model for Reaction Monitoring**
In this tutorial series we build a PLS regression model to monitor a chemical reaction in real time using Raman spectra. 

### **Tutorial 1.1. Loading and assmebling training data**
This notebook covers loading the calibration data and assembling it into a single DataFrame ready for modelling.

#### **A. Import spectral data**
Each spectrum is stored as a separate `.csv` file. The filename encodes the sample number and timestamp. We use a reader to parse all files in the folder and load them into a list of `Spectrum` objects.

In [ ]:
# Reader: parses filenames and loads all calibration spectra from a folder
from src.readers import read_calibration_spectra


In [9]:
path_to_training_data = r"datasets/tutorial1/spectra/calibration"

# Read all .csv files in the folder; sample_nr and datetime are parsed from the filename
spectra = read_calibration_spectra(path_to_training_data)


In [14]:
import pandas as pd

# Convert the list of Spectrum objects into a DataFrame
df = pd.DataFrame(spectra)

# Expand the metadata dict (sample_nr, datetime) into individual columns
df = pd.concat([df.drop(columns="metadata"), pd.json_normalize(df["metadata"])], axis=1)
df.head(5)


,x,y,sample_nr,datetime
0,"[400.0, 403.030303030303, 406.06060606060606, ...","[0.006122243514278804, 0.006123243669741856, 0...",1,2026-07-02 08:00:00
1,"[400.0, 403.030303030303, 406.06060606060606, ...","[0.0061391966052813515, 0.006140345134331623, ...",2,2026-07-02 08:04:08
2,"[400.0, 403.030303030303, 406.06060606060606, ...","[0.00615549718885005, 0.00615360264952341, 0.0...",3,2026-07-02 08:07:39
3,"[400.0, 403.030303030303, 406.06060606060606, ...","[0.006171254437689434, 0.0061698697395910366, ...",4,2026-07-02 08:11:26
4,"[400.0, 403.030303030303, 406.06060606060606, ...","[0.006184728364812914, 0.006185868038893945, 0...",5,2026-07-02 08:15:09


#### **B. Import reference data**
The reference file contains the known concentrations (`A` and `B`) for each calibration sample. We load it and join it with the spectral DataFrame on `sample_nr`.

In [15]:
path_to_reference_data = r"datasets/tutorial1/reference"

# Load reference data: columns are sample_nr, A (concentration), B (concentration)
df_reference = pd.read_csv(f"{path_to_reference_data}/calibration_data.csv")
df_reference.head(5)


,sample_nr,A,B
0,1,0.000000,0.465101
1,2,0.560012,5.787305
2,3,0.286100,12.686551
3,4,0.714314,18.845843
4,5,0.000000,25.514875


In [16]:
# Join spectra and reference data on sample_nr to get a single training DataFrame
df = df.merge(df_reference, on="sample_nr")
df.head(5)


,x,y,sample_nr,datetime,A,B
0,"[400.0, 403.030303030303, 406.06060606060606, ...","[0.006122243514278804, 0.006123243669741856, 0...",1,2026-07-02 08:00:00,0.000000,0.465101
1,"[400.0, 403.030303030303, 406.06060606060606, ...","[0.0061391966052813515, 0.006140345134331623, ...",2,2026-07-02 08:04:08,0.560012,5.787305
2,"[400.0, 403.030303030303, 406.06060606060606, ...","[0.00615549718885005, 0.00615360264952341, 0.0...",3,2026-07-02 08:07:39,0.286100,12.686551
3,"[400.0, 403.030303030303, 406.06060606060606, ...","[0.006171254437689434, 0.0061698697395910366, ...",4,2026-07-02 08:11:26,0.714314,18.845843
4,"[400.0, 403.030303030303, 406.06060606060606, ...","[0.006184728364812914, 0.006185868038893945, 0...",5,2026-07-02 08:15:09,0.000000,25.514875


#### **C. Save the training data**
We persist the assembled DataFrame as a Parquet file so it can be reused in the next notebooks without repeating the import steps.

In [13]:
# Save to Parquet for efficient reuse in downstream notebooks
df.to_parquet("local/tutorial_1_calibration_data.parquet", index=False)
